In [ ]:
from docplex.mp.model import Model
from collections import defaultdict
import itertools
from datetime import timedelta

# Smaller problem constants
MinUtilTime = 100
MaxUtilTime = 200
vehicle_fixed_cost = 100
HOURS = 4
nvehicle = 10
MINUTES_IN_DAY = 240
Terminals = ['A', 'B']
Depot = 'D'

# Reduced demand profile
freq1 = [0, 0, 1, 2]
freq2 = [0, 1, 1, 1]

arc_data = {
    ("D", "B"): (10, 10),
    ("D", "A"): (20, 10),
    ("B", "A"): (30, 1),
    ("A", "B"): (30, 1),
    ("B", "D"): (10, 10),
    ("A", "D"): (20, 10)
}

transitions = {"A": "B", "B": "A"}
travel_time = {("A", "B"): 30, ("B", "A"): 30}

# --- NODE and ARC GENERATION ---
nodes = []
node_id = 1
nodes.append({'id': node_id, 'time': 0, 'loc': Depot})
start_node_id = node_id
node_id += 1

for h in range(HOURS):
    for i in range(freq1[h]):
        t = h * 60 + (i * 60 // max(freq1[h], 1))
        nodes.append({'id': node_id, 'time': t, 'loc': Terminals[0]})
        node_id += 1
    for i in range(freq2[h]):
        t = h * 60 + (i * 60 // max(freq2[h], 1))
        nodes.append({'id': node_id, 'time': t, 'loc': Terminals[1]})
        node_id += 1

nodes.append({'id': node_id, 'time': MINUTES_IN_DAY, 'loc': Depot})
end_node_id = node_id
node_id += 1

nodes.sort(key=lambda x: x['time'])
V = nodes.copy()

E = []
for node in nodes:
    loc, t, nid = node['loc'], node['time'], node['id']
    if loc in Terminals:
        cost, cap = arc_data[(Depot, loc)]
        if t >= cost:
            E.append({'src': {'id': start_node_id, 'time': 0, 'loc': Depot},
                      'dst': {'id': nid, 'time': t, 'loc': loc},
                      'cost': cost, 'N_bus': cap})

for node in nodes:
    if node['loc'] not in transitions:
        continue
    curr_time = node['time']
    curr_loc = node['loc']
    src_id = node['id']

    next_loc = transitions[curr_loc]
    travel_t = travel_time[(curr_loc, next_loc)]
    dst_time = curr_time + travel_t

    if dst_time <= MINUTES_IN_DAY:
        dst_id = node_id
        node_id += 1
        V.append({'id': dst_id, 'time': dst_time, 'loc': next_loc})
        E.append({'src': {'id': src_id, 'time': curr_time, 'loc': curr_loc},
                  'dst': {'id': dst_id, 'time': dst_time, 'loc': next_loc},
                  'cost': travel_t, 'N_bus': 1})
        E.append({'src': {'id': dst_id, 'time': dst_time, 'loc': next_loc},
                  'dst': {'id': end_node_id, 'time': MINUTES_IN_DAY, 'loc': Depot},
                  'cost': arc_data[(next_loc, Depot)][0], 'N_bus': arc_data[(next_loc, Depot)][1]})


print(E)

# --- DUTY CONSTRUCTION ---
TRIP_NODES = sorted(set(e['src']['id'] for e in E if e['N_bus'] == 1))

DUTIES = []
def is_valid_arc(i, j):
    return any(e['src']['id'] == i and e['dst']['id'] == j for e in E)

def get_arc_cost(i, j):
    for e in E:
        if e['src']['id'] == i and e['dst']['id'] == j:
            return e['cost']
    return float('inf')

def duty_cost(duty):
    return vehicle_fixed_cost + sum(get_arc_cost(duty[i], duty[i+1]) for i in range(len(duty)-1))

def duty_time(duty):
    return sum(get_arc_cost(duty[i], duty[i+1]) for i in range(len(duty)-1)
               if any(e['src']['id'] == duty[i] and e['dst']['id'] == duty[i+1] and e['N_bus'] == 1 for e in E))

def duty_covers(duty):
    return set(n for n in duty if n in TRIP_NODES)

for r in range(1, 3):
    for trips in itertools.permutations(TRIP_NODES, r):
        if len(set(trips)) != len(trips):
            continue
        duty = [start_node_id] + list(trips) + [end_node_id]
        if not all(is_valid_arc(duty[i], duty[i+1]) for i in range(len(duty)-1)):
            continue
        if not (MinUtilTime <= duty_time(duty) <= MaxUtilTime):
            continue
        DUTIES.append(tuple(duty))

DUTY_COST = {d: duty_cost(d) for d in DUTIES}
DUTY_COVERS = {d: duty_covers(d) for d in DUTIES}

# --- MASTER LP ---
model = Model("VSP_SetPartitioning")
x = model.continuous_var_dict(DUTIES, name="x", lb=0, ub=1)

trip_constraints = {}
for nid in TRIP_NODES:
    covering = [d for d in DUTIES if nid in DUTY_COVERS[d]]
    if covering:
        trip_constraints[nid] = model.add_constraint(
            model.sum(x[d] for d in covering) == 1,
            ctname=f"cover_{nid}"
        )
    else:
        print(f"⚠️ No initial duty covers trip node {nid}. Skipping constraint.")

model.minimize(model.sum(DUTY_COST[d] * x[d] for d in DUTIES))

# --- PRICING SUBPROBLEM ---
def generate_new_duties(duals):
    new_duties = []
    for r in range(2, 4):
        for perm in itertools.permutations(TRIP_NODES, r):
            if len(set(perm)) != len(perm):
                continue
            duty = [start_node_id] + list(perm) + [end_node_id]
            if not all(is_valid_arc(duty[i], duty[i+1]) for i in range(len(duty)-1)):
                continue
            time = duty_time(duty)
            if not (MinUtilTime <= time <= MaxUtilTime):
                continue
            red_cost = duty_cost(duty) - sum(duals.get(n, 0) for n in duty_covers(duty))
            if red_cost < -1e-5:
                new_duties.append((tuple(duty), red_cost))
    return new_duties

# --- COLUMN GENERATION ---
while True:
    sol = model.solve()
    if not sol:
        print("❌ No feasible LP solution.")
        break

    duals = {n: trip_constraints[n].dual_value for n in trip_constraints}
    new_duties = generate_new_duties(duals)
    if not new_duties:
        print("✅ No improving duties. Optimal LP solution.")
        break

    for d, rc in new_duties:
        if d in x:
            continue
        x[d] = model.continuous_var(name=f"x_{d}", lb=0, ub=1)
        DUTY_COST[d] = duty_cost(d)
        DUTY_COVERS[d] = duty_covers(d)
        for nid in DUTY_COVERS[d]:
            trip_constraints[nid].lhs += x[d]
        print(f"➕ Added duty {d} with reduced cost {rc:.2f}")
        break

# --- OUTPUT ---
print("\nFinal LP Solution:")
if model.solution:
    print(f"Total cost: {model.objective_value:.2f}")
    for d in x:
        if x[d].solution_value > 1e-4:
            print(f"  Duty: {d}, cost = {DUTY_COST[d]}, x = {x[d].solution_value:.2f}")
else:
    print("No LP solution.")


In [7]:
from docplex.mp.model import Model
import itertools

# Define trips and dummy times
TRIPS = [1, 2, 3, 4, 5]
TRIP_TIMES = {
    1: (110, 'HSK', 'ATB'),
    2: (110, 'ATB', 'HSK'),
    3: (110, 'HSK', 'ATB'),
    4: (110, 'ATB', 'HSK'),
    5: (110, 'HSK', 'ATB'),
}

# Initial duties from real schedules
INITIAL_DUTIES = [
    (1, 2),       # from Vehicle 15
    (3, 4, 5),    # from Vehicle 18
]

# Cost constants
VEHICLE_COST = 50
CONNECT_COST = 2  # Between trips
def duty_cost(duty):
    return VEHICLE_COST + (len(duty)-1)*CONNECT_COST + sum(TRIP_TIMES[t][0] for t in duty)

def duty_covers(duty):
    return set(duty)

# Setup model
model = Model("MiniColumnGen")
x = model.continuous_var_dict(INITIAL_DUTIES, name="x", lb=0, ub=1)

# Constraints: each trip must be covered exactly once
trip_constraints = {}
for t in TRIPS:
    trip_constraints[t] = model.add_constraint(
        model.sum(x[d] for d in INITIAL_DUTIES if t in duty_covers(d)) == 1,
        ctname=f"cover_{t}"
    )

# Objective
model.minimize(model.sum(duty_cost(d) * x[d] for d in INITIAL_DUTIES))

# Pricing: generate duties with negative reduced cost
def generate_new_duties(duals):
    new_duties = []
    for r in range(2, 4):
        for perm in itertools.permutations(TRIPS, r):
            if list(perm) in INITIAL_DUTIES: continue
            cost = duty_cost(perm)
            reduced_cost = cost - sum(duals.get(t, 0) for t in perm)
            if reduced_cost < -1e-5:
                new_duties.append((perm, reduced_cost))
    return new_duties

# Column generation loop
while True:
    sol = model.solve()
    if not sol:
        print("❌ No solution.")
        break

    duals = {t: trip_constraints[t].dual_value for t in TRIPS}
    new_duties = generate_new_duties(duals)

    if not new_duties:
        print("✅ Optimal integer solution found.")
        break

    for d, rc in new_duties:
        if d in x:
            continue
        x[d] = model.continuous_var(name=f"x_{d}", lb=0, ub=1)
        INITIAL_DUTIES.append(d)
        for t in d:
            trip_constraints[t].lhs += x[d]
        print(f"➕ Added new duty {d} with reduced cost {rc:.2f}")
        break

# Output
print("\nFinal IP Solution:")
if model.solution:
    print(f"Total cost: {model.objective_value:.2f}")
    for d in x:
        if x[d].solution_value > 1e-4:
            print(f"  Duty: {d}, cost = {duty_cost(d)}, x = {x[d].solution_value:.2f}")


➕ Added new duty (1, 3) with reduced cost -384.00
➕ Added new duty (2, 3) with reduced cost -768.00
➕ Added new duty (1, 4) with reduced cost -384.00
➕ Added new duty (1, 5) with reduced cost -520.00
➕ Added new duty (2, 4) with reduced cost -1312.00
✅ Optimal integer solution found.

Final IP Solution:
Total cost: 128.00
  Duty: (3, 4, 5), cost = 384, x = 0.33
  Duty: (2, 3), cost = 272, x = 0.67
  Duty: (1, 4), cost = 272, x = 0.33
  Duty: (1, 5), cost = 272, x = 0.67
  Duty: (2, 4), cost = 272, x = 0.33
